## CODING DRILL: Manual XGBoost-Style Split Evaluation (Log-Loss)

**Scenario:** You're implementing the core split-finding logic of a gradient-boosted tree from scratch — no library — for a binary classification problem using log-loss. This is exactly the kind of "build it from primitives" exercise that separates candidates who've called `.fit()` from ones who understand what's inside it.

### The Task

You have 8 training samples at boosting round $t$, with true labels $y_i$ and current ensemble predictions (as probabilities) $p_i^{(t-1)}$:

| i | $y_i$ | $p_i^{(t-1)}$ | feature $x$ |
|---|---|---|---|
| 0 | 1 | 0.55 | 2.1 |
| 1 | 0 | 0.48 | 3.4 |
| 2 | 1 | 0.62 | 1.2 |
| 3 | 0 | 0.30 | 5.6 |
| 4 | 1 | 0.71 | 1.8 |
| 5 | 0 | 0.20 | 6.1 |
| 6 | 1 | 0.58 | 2.5 |
| 7 | 0 | 0.45 | 4.0 |

**Implement the following, from scratch (no `xgboost`/`sklearn` — raw Python/NumPy only):**

1. Compute $g_i$ and $h_i$ for each sample under log-loss, where:
   - $g_i = p_i - y_i$
   - $h_i = p_i(1-p_i)$

2. Write a function `leaf_weight(G, H, lam)` implementing $w^* = -G/(H+\lambda)$.

3. Write a function `split_gain(G_L, H_L, G_R, H_R, lam, gamma)` implementing the exact Gain formula from our lesson.

4. Using `feature x`, find the **best split threshold** among the candidate midpoints between sorted consecutive x-values, by computing Gain for every candidate split and returning the one that maximizes Gain. Use $\lambda = 1.0$, $\gamma = 0.0$.

5. Print: the chosen split threshold, the resulting `w_left` and `w_right`, and the Gain achieved.

**Constraints / what I'm grading:**
- No shortcuts through `xgboost`, `lightgbm`, or `sklearn.tree`
- Vectorize where sensible, but the split-search loop can be a plain Python loop (production XGBoost uses sorted-histogram search — I'll ask you about that separately)
- Handle the edge case: what if $H_L + \lambda = 0$ or $H_R + \lambda = 0$? Your code shouldn't crash on a degenerate split.

Write your implementation. When you're done, paste the code and I'll review it for correctness, mathematical fidelity to the formulas, and production-readiness (vectorization, edge cases, numerical stability) — then give you the scorecard and gold-standard solution.

In [ ]:
import numpy as np

# ============================================================
# INPUT DATA
# ============================================================
y = np.array([1, 0, 1, 0, 1, 0, 1, 0], dtype=float)
p = np.array([0.55, 0.48, 0.62, 0.30, 0.71, 0.20, 0.58, 0.45], dtype=float)
x = np.array([2.1, 3.4, 1.2, 5.6, 1.8, 6.1, 2.5, 4.0], dtype=float)

LAMBDA = 1.0
GAMMA = 0.0
EPS = 1e-12  # for numerical stability guards


# ============================================================
# STEP 1: Gradients and Hessians (log-loss)
# ============================================================
def compute_gradients_hessians(y, p):
    """
    Returns g_i, h_i for each sample under log-loss.
    g_i = p_i - y_i
    h_i = p_i * (1 - p_i)
    """
    if len(y) != len(p):
        raise ValueError(f"Lengths do not match")
    g_i = np.subtract(p,y)
    h_i = np.multiply(p,np.subtract(1,p))
    return (g_i,h_i)


# ============================================================
# STEP 2: Closed-form leaf weight
# ============================================================
def leaf_weight(G, H, lam):
    """
    w* = -G / (H + lam)
    Must not crash if H + lam == 0.
    """
    if G and (H+lam) == 0:
        print("Race condition")
        return 0
    elif H+lam == 0:
        print(" Second Race condition")
        return 0
    else:
        return - (G/(H+lam))


# ============================================================
# STEP 3: Split gain formula
# ============================================================
def split_gain(G_L, H_L, G_R, H_R, lam, gamma):
    """
    Gain = 0.5 * [ G_L^2/(H_L+lam) + G_R^2/(H_R+lam)
                   - (G_L+G_R)^2/(H_L+H_R+lam) ] - gamma
    Must not crash on degenerate denominators.
    """
    left_leaf_score = G_L**2/(H_L + lam)
    right_leaf_score = G_R**2/(H_R + lam)
    unsplit_score = (G_L+G_R)**2/(H_L+H_R+lam)
    gain = 0.5*((left_leaf_score + right_leaf_score - unsplit_score)-gamma)
    return gain


# ============================================================
# STEP 4: Best split search over candidate thresholds
# ============================================================
def find_best_split(x, g, h, lam, gamma):
    """
    1. Sort samples by x.
    2. Candidate thresholds = midpoints between consecutive sorted x values.
    3. For each candidate threshold, partition samples into left/right,
       compute G_L, H_L, G_R, H_R, then Gain.
    4. Track and return the threshold with max Gain, along with
       G_L, H_L, G_R, H_R at that split.
    """
    # Hash Variables
    max_gain = -np.inf
    result_set = {}
    # Sort based on x
    x,g,h = zip(*sorted(zip(x,g,h)))
    x,g,h = np.array(x),np.array(g),np.array(h)
    # calculate thresholds
    thresholds = []
    for i in range(len(x)-1):
        thresholds.append((x[i]+x[i+1])/2)
    for t in range(len(thresholds)):
        L_group = [(x, y, z) for x, y, z in zip(x,g,h) if x > t]
        R_group = [(x, y, z) for x, y, z in zip(x,g,h) if x <= t]
        L_x,L_g,L_h = zip(*L_group) if L_group else ([], [], [])
        R_x,R_g,R_h = zip(*R_group) if R_group else ([], [], [])
        L_x,L_g,L_h = np.array(L_x),np.array(L_g),np.array(L_h)
        R_x,R_g,R_h = np.array(R_x),np.array(R_g),np.array(R_h)
        G_L = np.sum(L_g)
        H_L = np.sum(L_h)
        G_R = np.sum(R_g)
        H_R = np.sum(R_h)
        # Calculate the gain
        gain = split_gain(G_L,H_L,G_R,H_R,lam,gamma)
        if gain > max_gain:
            print(f"Found max gain: {gain} while threshold is :{t}")
            result_set = {
                            'threshold':t,
                            'max_gain':gain,
                            'G_L':G_L,
                            'H_L':H_L,
                            'G_R':G_R,
                            'H_R':H_R
                        }
            max_gain = gain
    return result_set






# ============================================================
# STEP 5: Driver
# ============================================================
def main():
    # TODO:
    # 1. compute g, h
    g_i,h_i = compute_gradients_hessians(y,p)
    print(f"The gradients are : {g_i}")
    print(f"The hessians are : {h_i}")
    # 2. run find_best_split
    best_split = find_best_split(x,g_i,h_i,LAMBDA,GAMMA)
    # 3. compute w_left, w_right via leaf_weight
    w_left = leaf_weight(G=best_split['G_L'],H=best_split['H_L'],lam=LAMBDA)
    w_right = leaf_weight(G=best_split['G_R'],H=best_split['H_R'],lam=LAMBDA)
    # 4. print threshold, w_left, w_right, gain
    print(f"The optimal threshold is at: {best_split['threshold']}")
    print(f"The gain at t = {best_split['threshold']} is at: {best_split['max_gain']}")
    print(f"The left leaf weight is: {w_left}")
    print(f"The right leaf weight is: {w_right}")

if __name__ == "__main__":
    main()

The gradients are : [-0.45  0.48 -0.38  0.3  -0.29  0.2  -0.42  0.45]
The hessians are : [0.2475 0.2496 0.2356 0.21   0.2059 0.16   0.2436 0.2475]
Found max gain at : 0.0 while threshold is :0
Found max gain at : 0.22003630787319756 while threshold is :2
Found max gain at : 1.1590305648502852 while threshold is :3
The optimal threshold is at: 3
The gain at t = 3 is at: 1.1590305648502852
The left leaf weight is: -0.7658936318354668
The right leaf weight is: 0.7968539790955189


In [21]:
import numpy as np
x = [3,1,2]
p = [1,2,9]
g = [0,1,1]

np.subtract(1,p)

for i in zip(*sorted(zip(x, p, g))):
    print(np.array(i))

for i in range(len(x)-1):
        print((x[i]+x[i+1])/2)

t = 2
L_group1 = [(x, y, z) for x, y, z in zip(x,g,p) if x > t]
R_group2 = [(x, y, z) for x, y, z in zip(x,g,p) if x <= t]

print(f"Left: {L_group1}")

# l1_high, l2_high, l3_high = zip(*group1) if group1 else ([], [], [])
# l1_low, l2_low, l3_low = zip(*group2) if group2 else ([], [], [])

l_x,l_y,l_z = zip(*L_group1) if L_group1 else ([], [], [])

print(l_x,l_y,l_z)

[1 2 3]
[2 9 1]
[1 1 0]
2.0
1.5
Left: [(3, 0, 1)]
(3,) (0,) (1,)
